# Text-to-Speech synthesis — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def cosine(a,b):
    a=np.asarray(a,dtype=float); b=np.asarray(b,dtype=float); return float(a@b/(np.linalg.norm(a)*np.linalg.norm(b)))
def hz_to_mel(f): return 2595*np.log10(1+np.asarray(f)/700)
def mel_to_hz(m): return 700*(10**(np.asarray(m)/2595)-1)
def stft_mag(x,N=256,H=64):
    w=np.hanning(N); frames=[]
    for s in range(0,len(x)-N+1,H): frames.append(np.abs(np.fft.rfft(x[s:s+N]*w)))
    return np.array(frames).T
print('setup ok')

## Duration, prosody, and acoustic-frame prediction
A duration-based TTS toy shows text expansion, pitch control, mel loss, waveform rendering, and a tiny text-to-acoustics pass.

In [ ]:
# 1) token durations expand text states into acoustic frames
tokens=list('helo'); dur=np.array([2,1,2,1]); expanded=sum(([tok]*d for tok,d in zip(tokens,dur)),[]); starts=np.r_[0,np.cumsum(dur)[:-1]]
plt.figure(figsize=(6,3)); plt.bar(tokens,dur); plt.ylabel('frames'); plt.title(f'total frames={dur.sum()}')
assert dur.sum()==6 and starts.tolist()==[0,2,3,5] and expanded==['h','h','e','l','l','o']

In [ ]:
# 2) pitch contour controls prosody over frames
f0=np.array([180,190,210,200,170,160]); frames=np.arange(len(f0))
plt.figure(figsize=(6,3)); plt.plot(frames,f0,'-o'); plt.xlabel('frame'); plt.ylabel('Hz'); plt.title(f'mean={f0.mean():.1f}, range={f0.max()-f0.min()}')
assert abs(f0.mean()-185.0)<1e-9 and f0.max()-f0.min()==50

In [ ]:
# 3) weighted acoustic loss trades mel and pitch errors
mel_pred=np.array([0.7,0.2,0.4]); mel_true=np.array([1.0,0.1,0.6]); f_pred=np.array([190.]); f_true=np.array([180.]); lam=.01; mel_l1=np.mean(np.abs(mel_pred-mel_true)); pitch=lam*np.mean(np.abs(f_pred-f_true)); loss=mel_l1+pitch
plt.figure(figsize=(6,3)); plt.bar(['mel L1','weighted pitch','total'],[mel_l1,pitch,loss]); plt.title('two acoustic terms share one loss')
assert abs(mel_l1-0.2)<1e-9 and abs(loss-0.3)<1e-9

In [ ]:
# 4) a simple vocoder-like sine follows predicted f0
sr=8000; f0=np.array([180,190,210,200,170,160]); samples=[]
for f in f0: samples.append(np.sin(2*np.pi*f*np.arange(160)/sr))
wav=np.concatenate(samples)
plt.figure(figsize=(6,3)); plt.plot(wav[:500]); plt.title('rendered toy waveform follows f0')
assert len(wav)==960 and np.max(np.abs(wav))<=1.0

In [ ]:
# 5) end-to-end toy text to mel frames through durations
tokens=list('helo'); dur=np.array([2,1,2,1]); table={'h':[1,0,0],'e':[0,1,0],'l':[0,0.6,0.4],'o':[0,0,1]}; mel=np.array([table[t] for t in sum(([tok]*d for tok,d in zip(tokens,dur)),[])])
plt.figure(figsize=(6,3)); plt.imshow(mel.T,aspect='auto',origin='lower'); plt.xlabel('frame'); plt.ylabel('mel bin'); plt.title('expanded tokens become acoustic frames')
assert mel.shape==(6,3) and np.allclose(mel[0],[1,0,0]) and np.allclose(mel[-1],[0,0,1])